In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk

FILE_PATH = "/path/to/your/data/dice_all_fold.xlsx"
pf_file = "/path/to/your/data/Dataset/Spiderdataset/radiological_gradingscsv"

GT_DIR = "/path/to/your/data/Spiderdataset/masks"
PRED_DIR = "/path/to/your/data/Prediction_all_fold"
OUTPUT_DIR = "/path/to/your/data/Dice_Verification"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
gt_index = {}

for filepath in glob.glob(os.path.join(GT_DIR, "*_t2.mha")):

    filename = os.path.basename(filepath)

    # Exact match: 13_t2.mha
    # This will NOT match 13_t2_SPACE.mha
    m = re.match(r"(\d+)_t2\.mha$", filename, re.IGNORECASE)

    if m:
        case_id = m.group(1)
        gt_index[case_id] = filepath

print("T2 GT files found:", len(gt_index))

In [ ]:
# Definitive GT -> Prediction label mapping

LABEL_MAPPING = {
    1: 1,       # L5
    2: 2,       # L4
    3: 3,       # L3
    4: 4,       # L2
    5: 5,       # L1

    201: 6,     # L5/S1
    202: 7,     # L4/L5
    203: 8,     # L3/L4
    204: 9,     # L2/L3
    205: 10,    # L1/L2
    206: 11     # T12/L1
}

STRUCTURE_NAMES = {
    1: "L5",
    2: "L4",
    3: "L3",
    4: "L2",
    5: "L1",
    201: "L5/S1",
    202: "L4/L5",
    203: "L3/L4",
    204: "L2/L3",
    205: "L1/L2",
    206: "T12/L1"
}

In [ ]:
#Structure-level failure
threshold = 0.70

failure_rows = dice_df[dice_df["Dice"] < threshold].copy()

print("Total failed structure rows:", len(failure_rows))

print(
    failure_rows[
        ["case_id", "label_id", "structure", "Dice"]
    ]
)

In [ ]:
#Case-level failure
threshold = 0.70

# Find individual failed structures
failure_rows = dice_df[dice_df["Dice"] < threshold].copy()

# Find unique cases containing at least one failure
failure_cases = (
    failure_rows["case_id"]
    .drop_duplicates()
    .sort_values()
)

print(
    "Number of failure cases:",
    len(failure_cases)
)

print(
    "Failure cases:",
    failure_cases.tolist()
)

In [ ]:
import pandas as pd

# ============================================================
# INPUT FILE
# ============================================================

file_path = "/path/to/your/data/Dice_Verification/Dice_Verification_46_Cases.xlsx"

# OUTPUT FILE
output_path = "/path/to/your/data/failure_cases_wide_format.xlsx"

# ============================================================
# READ DATA
# ============================================================

df = pd.read_excel(file_path)

df.columns = df.columns.str.strip()

df["case_id"] = df["case_id"].astype(str).str.strip()
df["label_id"] = pd.to_numeric(df["label_id"], errors="coerce")
df["Dice"] = pd.to_numeric(df["Dice"], errors="coerce")

# ============================================================
# FINAL LABEL MAPPING
# ============================================================

label_to_class = {
    1: "Class_1",
    2: "Class_2",
    3: "Class_3",
    4: "Class_4",
    5: "Class_5",
    201: "Class_6",
    202: "Class_7",
    203: "Class_8",
    204: "Class_9",
    205: "Class_10",
    206: "Class_11"
}

class_order = [
    "Class_1",
    "Class_2",
    "Class_3",
    "Class_4",
    "Class_5",
    "Class_6",
    "Class_7",
    "Class_8",
    "Class_9",
    "Class_10",
    "Class_11"
]

# Map label ID → Class
df["Class"] = df["label_id"].map(label_to_class)

# ============================================================
# CONVERT LONG → WIDE
# ============================================================

wide_df = df.pivot_table(
    index="case_id",
    columns="Class",
    values="Dice",
    aggfunc="first"
).reset_index()

# Make sure all classes are present
for class_name in class_order:
    if class_name not in wide_df.columns:
        wide_df[class_name] = None

# Correct column order
wide_df = wide_df[
    ["case_id"] + class_order
]

# ============================================================
# FIND FAILED CLASSES
# ============================================================

threshold = 0.70

def get_failed_classes(row):
    return [
        class_name
        for class_name in class_order
        if pd.notna(row[class_name])
        and row[class_name] < threshold
    ]

wide_df["Failed_Classes"] = wide_df.apply(
    get_failed_classes,
    axis=1
)

wide_df["Num_failed_classes"] = (
    wide_df["Failed_Classes"].apply(len)
)

# ============================================================
# KEEP ONLY FAILURE CASES
# ============================================================

failure_cases = wide_df[
    wide_df["Num_failed_classes"] > 0
].copy()

# ============================================================
# ADD FOLD
# ============================================================

# If Fold exists in the input file
if "Fold" in df.columns:

    fold_info = (
        df.groupby("case_id")["Fold"]
        .first()
        .reset_index()
    )

    failure_cases = failure_cases.merge(
        fold_info,
        on="case_id",
        how="left"
    )

else:
    failure_cases["Fold"] = ""

# ============================================================
# RENAME CASE
# ============================================================

failure_cases.rename(
    columns={"case_id": "Case"},
    inplace=True
)

# ============================================================
# FINAL COLUMN ORDER
# ============================================================

failure_cases = failure_cases[
    [
        "Case",
        "Fold",
        "Class_1",
        "Class_2",
        "Class_3",
        "Class_4",
        "Class_5",
        "Class_6",
        "Class_7",
        "Class_8",
        "Class_9",
        "Class_10",
        "Class_11",
        "Num_failed_classes",
        "Failed_Classes"
    ]
]

# ============================================================
# SAVE EXCEL FILE
# ============================================================

failure_cases.to_excel(
    output_path,
    index=False
)

# ============================================================
# PRINT SUMMARY
# ============================================================

print("============================================")
print("Failure file created successfully")
print("============================================")

print("Total failure cases:", len(failure_cases))
print("Output file:")
print(output_path)

print("\nPreview:")
print(failure_cases.to_string(index=False))

In [ ]:
# ============================================================
# CALCULATE MEAN DICE FOR VERTEBRAE AND IVD (Structure-level)
# ============================================================

import pandas as pd

input_file = "/path/to/your/data/Dice_Verification/Dice_Verification_46_Cases.xlsx"

# Read the already calculated Dice results
df = pd.read_excel(input_file)

# Make sure Dice is numeric
df["Dice"] = pd.to_numeric(df["Dice"], errors="coerce")

# ------------------------------------------------------------
# Vertebrae = labels 1–5
# IVD = labels 201–206
# ------------------------------------------------------------

vertebra_df = df[df["label_id"].isin([1, 2, 3, 4, 5])]

ivd_df = df[df["label_id"].isin([201, 202, 203, 204, 205, 206])]

# ------------------------------------------------------------
# Overall mean Dice
# ------------------------------------------------------------

mean_vertebra_dice = vertebra_df["Dice"].mean()
mean_ivd_dice = ivd_df["Dice"].mean()

print("\n======================================")
print("MEAN DICE")
print("======================================")

print(f"Mean Vertebra Dice : {mean_vertebra_dice:.6f}")
print(f"Mean IVD Dice      : {mean_ivd_dice:.6f}")

In [ ]:
#Calculate Physical spacing
img = sitk.ReadImage(gt_path)

gt = sitk.GetArrayFromImage(img)
spacing_xyz = img.GetSpacing()

spacing_zyx = spacing_xyz[::-1]

In [ ]:
from medpy.metric import binary

def calculate_metrics(gt_bin, pred_bin, spacing):

    # Both masks empty
    if not gt_bin.any() and not pred_bin.any():
        return np.nan, np.nan, np.nan

    # GT exists but prediction is completely absent
    if gt_bin.any() and not pred_bin.any():
        return 0.0, np.inf, np.inf

    # Prediction exists but GT is absent
    if not gt_bin.any() and pred_bin.any():
        return np.nan, np.nan, np.nan

    dice = binary.dc(pred_bin, gt_bin)

    hd95 = binary.hd95(
        pred_bin,
        gt_bin,
        voxelspacing=spacing,
        connectivity=1
    )

    asd = binary.asd(
        pred_bin,
        gt_bin,
        voxelspacing=spacing,
        connectivity=1
    )

    return dice, hd95, asd

In [ ]:
results = []

for fname in sorted(os.listdir(GT_DIR)): # Use GT_DIR

    # Change to check for .mha files
    if not fname.endswith(".mha"):
        continue

    case_id = fname.split('_')[0] # Extract case_id from filename

    gt_path = os.path.join(GT_DIR, fname)
    # Assuming prediction files match GT filenames by case_id, but might have different extensions
    # Reconstruct pred_path based on case_id and pred_index to ensure it matches the actual prediction file
    if case_id not in pred_index:
        print(f"Prediction missing for case {case_id}: {fname}")
        continue
    pred_path = pred_index[case_id]

    gt_img = sitk.ReadImage(gt_path)
    pred_img = sitk.ReadImage(pred_path)

    gt = sitk.GetArrayFromImage(gt_img)
    pred = sitk.GetArrayFromImage(pred_img)

    # Add a check for shape compatibility to prevent ValueError
    if gt.shape != pred.shape:
        print(f"WARNING: Shape mismatch for case {case_id}: GT shape {gt.shape} vs Pred shape {pred.shape}. Skipping.")
        continue

    # Physical spacing corresponding to NumPy array axes
    spacing = gt_img.GetSpacing()[::-1]

    for gt_label_id, pred_label_id in LABEL_MAPPING.items(): # Use LABEL_MAPPING and iterate correctly

        gt_bin = (gt == gt_label_id)
        pred_bin = (pred == pred_label_id)

        dice, hd95, asd = calculate_metrics(
            gt_bin,
            pred_bin,
            spacing
        )

        # Get structure name from STRUCTURE_NAMES using the GT label ID
        structure_name = STRUCTURE_NAMES.get(gt_label_id, "Unknown")

        results.append({
            "case": case_id, # Store case_id explicitly
            "label": gt_label_id, # Store GT label ID
            "structure": structure_name,
            "Dice": dice,
            "HD95_mm": hd95,
            "ASD_mm": asd,
            "spacing_x": gt_img.GetSpacing()[0],
            "spacing_y": gt_img.GetSpacing()[1],
            "spacing_z": gt_img.GetSpacing()[2]
        })

df = pd.DataFrame(results)

print("Total rows:", len(df))
df.head()

In [ ]:
#Statistical analysics of Physical spacing
def percentile(x, q):
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    return np.percentile(x, q)


summary = []

for (structure, level), group in df.groupby(["structure", "level"]):

    hd = group["HD95_mm"].dropna()
    hd = hd[np.isfinite(hd)]

    asd = group["ASD_mm"].dropna()
    asd = asd[np.isfinite(asd)]

    summary.append({
        "Structure": structure,
        "Level": level,

        "N": len(group),

        "HD95 Median (mm)": np.median(hd) if len(hd) else np.nan,
        "HD95 Q1 (mm)": np.percentile(hd, 25) if len(hd) else np.nan,
        "HD95 Q3 (mm)": np.percentile(hd, 75) if len(hd) else np.nan,
        "HD95 IQR (mm)": (
            np.percentile(hd, 75) - np.percentile(hd, 25)
            if len(hd) else np.nan
        ),
        "HD95 P90 (mm)": percentile(hd, 90),
        "HD95 P95 (mm)": percentile(hd, 95),
        "HD95 P99 (mm)": percentile(hd, 99),

        "ASD Median (mm)": np.median(asd) if len(asd) else np.nan,
        "ASD Q1 (mm)": np.percentile(asd, 25) if len(asd) else np.nan,
        "ASD Q3 (mm)": np.percentile(asd, 75) if len(asd) else np.nan,
        "ASD IQR (mm)": (
            np.percentile(asd, 75) - np.percentile(asd, 25)
            if len(asd) else np.nan
        ),
        "ASD P90 (mm)": percentile(asd, 90),
        "ASD P95 (mm)": percentile(asd, 95),
        "ASD P99 (mm)": percentile(asd, 99),
    })

summary_df = pd.DataFrame(summary)

summary_df

Save everything

In [ ]:
output_path = "/path/to/your/workspace/HD95_ASD_structure_level_results.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="Individual_Metrics",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

print("Saved:", output_path)

## Wilcoxon signed-rank test

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

# --------------------------------------------------
# 1. Load the per-patient Dice data
# --------------------------------------------------

file_path ="/path/to/your/data/Original_dice_all_fold.xlsx"

# Use pd.read_excel for .xlsx files
df = pd.read_excel(file_path)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

In [ ]:
# --------------------------------------------------
# 2. Define vertebra and IVD classes
# --------------------------------------------------

vertebra_cols = [
    "Class_1",
    "Class_2",
    "Class_3",
    "Class_4",
    "Class_5"
]

ivd_cols = [
    "Class_6",
    "Class_7",
    "Class_8",
    "Class_9",
    "Class_10",
    "Class_11"
]

# Check that all columns exist
print("Missing vertebra columns:",
      [c for c in vertebra_cols if c not in df.columns])

print("Missing IVD columns:",
      [c for c in ivd_cols if c not in df.columns])

Calculate patient-level mean vertebral and IVD Dice

In [ ]:
# --------------------------------------------------
# 3. Patient-level mean Dice
# --------------------------------------------------

df["mean_vertebra_dice"] = df[vertebra_cols].mean(axis=1, skipna=True)

df["mean_ivd_dice"] = df[ivd_cols].mean(axis=1, skipna=True)

print(
    df[
        ["mean_vertebra_dice", "mean_ivd_dice"]
    ].head()
)

In [ ]:
# --------------------------------------------------
# 4. Keep patients with both measurements
# --------------------------------------------------

paired = df[
    ["mean_vertebra_dice", "mean_ivd_dice"]
].dropna()

print("Number of paired patients:", len(paired))

Wilcoxon signed-rank test

In [ ]:
# --------------------------------------------------
# 5. Wilcoxon signed-rank test
# --------------------------------------------------

stat, p_value = wilcoxon(
    paired["mean_vertebra_dice"],
    paired["mean_ivd_dice"],
    alternative="two-sided"
)

print("Wilcoxon signed-rank statistic:", stat)
print("p-value:", p_value)

patient-level descriptive values

In [ ]:
# --------------------------------------------------
# 6. Descriptive statistics for patient-level means
# --------------------------------------------------

print("\nPatient-level vertebral Dice:")
print("Mean:", paired["mean_vertebra_dice"].mean())
print("SD:", paired["mean_vertebra_dice"].std())
print("Median:", paired["mean_vertebra_dice"].median())

print("\nPatient-level IVD Dice:")
print("Mean:", paired["mean_ivd_dice"].mean())
print("SD:", paired["mean_ivd_dice"].std())
print("Median:", paired["mean_ivd_dice"].median())

paired difference

In [ ]:
# --------------------------------------------------
# 7. Paired difference
# --------------------------------------------------

paired["difference"] = (
    paired["mean_vertebra_dice"]
    - paired["mean_ivd_dice"]
)

print("\nMean paired difference:",
      paired["difference"].mean())

print("Median paired difference:",
      paired["difference"].median())

print("SD paired difference:",
      paired["difference"].std())

calculate the effect size

In [ ]:
import numpy as np

x = paired["mean_vertebra_dice"].to_numpy()
y = paired["mean_ivd_dice"].to_numpy()

differences = x - y

# Remove zero differences
differences_nonzero = differences[differences != 0]

n = len(differences_nonzero)

# Absolute differences and ranks
abs_diff = np.abs(differences_nonzero)

ranks = pd.Series(abs_diff).rank(method="average").to_numpy()

W_plus = ranks[differences_nonzero > 0].sum()
W_minus = ranks[differences_nonzero < 0].sum()

rank_biserial = (W_plus - W_minus) / (W_plus + W_minus)

print("Rank-biserial correlation:", rank_biserial)

In [ ]:
df["n_vertebra"] = df[vertebra_cols].notna().sum(axis=1)
df["n_ivd"] = df[ivd_cols].notna().sum(axis=1)

print(df["n_vertebra"].value_counts().sort_index())
print(df["n_ivd"].value_counts().sort_index())

In [ ]:
print(
    df[
        ["n_vertebra", "n_ivd"]
    ].describe()
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# --------------------------------------------------
# Keep paired observations
# --------------------------------------------------

paired = df[
    ["mean_vertebra_dice", "mean_ivd_dice"]
].dropna().copy()

print("Number of paired patients:", len(paired))

# --------------------------------------------------
# Wilcoxon signed-rank test
# --------------------------------------------------

stat, p_value = wilcoxon(
    paired["mean_vertebra_dice"],
    paired["mean_ivd_dice"],
    alternative="two-sided"
)

print("Wilcoxon statistic:", stat)
print("p-value:", p_value)


## Pfirrmann Failure Analysis

1. Load file path

In [ ]:
import pandas as pd
import numpy as np

dice_file = "/path/to/your/data/Original_dice_all_fold.xlsx"
pf_file = "/path/to/your/data/Dataset/Spiderdataset/radiological_gradings.csv"

dice_df = pd.read_excel(dice_file)
pf_df = pd.read_csv(pf_file)

print("Dice shape:", dice_df.shape)
print("Pfirrmann shape:", pf_df.shape)

print("\nDice columns:")
print(dice_df.columns.tolist())

print("\nPfirrmann columns:")
print(pf_df.columns.tolist())

In [ ]:
dice_df["patient"] = dice_df["Case"].apply(extract_patient_id)

3. Identify the Dice columns

In [ ]:
ivd_classes = [
    "Class_6",
    "Class_7",
    "Class_8",
    "Class_9",
    "Class_10",
    "Class_11"
]

missing = [c for c in ivd_classes if c not in dice_df.columns]

print("Missing IVD columns:", missing)

if missing:
    raise ValueError("Check your Dice column names.")

4.Standardize the patient/case identifier

In [ ]:
# Make patient numeric
pf_df["patient"] = pd.to_numeric(
    pf_df["patient"],
    errors="coerce"
)

print("\nPfirrmann patient identifiers:")
print(pf_df["patient"].head(20))

print("\nUnique Pfirrmann patients:")
print(pf_df["patient"].nunique())

IVD class mapping

In [ ]:
# --------------------------------------------------
# IVD Dice classes
# --------------------------------------------------

ivd_classes = [
    "Class_6",
    "Class_7",
    "Class_8",
    "Class_9",
    "Class_10",
    "Class_11"
]

missing = [
    c for c in ivd_classes
    if c not in dice_df.columns
]

print("Missing IVD columns:", missing)

if missing:
    raise ValueError("Check your Dice column names.")

In [ ]:
# --------------------------------------------------
# Confirm Dice class → anatomical disc mapping
# --------------------------------------------------

disc_map = {
    "Class_6": "L5_S1",
    "Class_7": "L4_L5",
    "Class_8": "L3_L4",
    "Class_9": "L2_L3",
    "Class_10": "L1_L2",
    "Class_11": "T12_L1"
}

print(
    pd.DataFrame(
        list(disc_map.items()),
        columns=["class_label", "disc_level"]
    )
)

5. Convert the six IVD classes into long format

In [ ]:
# --------------------------------------------------
# Convert IVD Dice from wide → long format
# --------------------------------------------------

dice_ivd = dice_df[
    ["patient"] + ivd_classes
].copy()

dice_ivd[ivd_classes] = dice_ivd[ivd_classes].apply(
    pd.to_numeric,
    errors="coerce"
)

dice_long = dice_ivd.melt(
    id_vars="patient",
    value_vars=ivd_classes,
    var_name="class_label",
    value_name="dice"
)

dice_long["disc_level"] = (
    dice_long["class_label"].map(disc_map)
)

print("Long-format Dice shape:", dice_long.shape)

print(dice_long.head(12))

6. Map Pfirrmann IVD labels

In [ ]:
# --------------------------------------------------
# Pfirrmann IVD label → anatomical disc
# --------------------------------------------------

pf_df["ivd_label"] = pd.to_numeric(
    pf_df["ivd_label"],
    errors="coerce"
)

pf_df["disc_level"] = pf_df["ivd_label"].map({
    1: "T12_L1",
    2: "L1_L2",
    3: "L2_L3",
    4: "L3_L4",
    5: "L4_L5",
    6: "L5_S1"
})

pf_df["pfirrman_grade"] = pd.to_numeric(
    pf_df["pfirrman_grade"],
    errors="coerce"
)

print(
    pf_df[
        [
            "patient",
            "ivd_label",
            "disc_level",
            "pfirrman_grade"
        ]
    ].head(20)
)

8. Merge Dice + Pfirrmann

In [ ]:
# --------------------------------------------------
# Merge Dice and Pfirrmann
# --------------------------------------------------

pf_small = pf_df[
    [
        "patient",
        "disc_level",
        "pfirrman_grade"
    ]
].copy()

merged = dice_long.merge(
    pf_small,
    on=["patient", "disc_level"],
    how="left"
)

print("Merged shape:", merged.shape)

print(
    merged.head(20)
)

11. Define segmentation failure

In [ ]:
merged["failure"] = (
    merged["dice"] < 0.70
)

print(
    "Total IVD observations:",
    len(merged)
)

print(
    "Total IVD failures:",
    merged["failure"].sum()
)

print(
    "Overall IVD failure rate:",
    merged["failure"].mean() * 100,
    "%"
)

12. Failure rate by Pfirrmann grade

In [ ]:
failure_summary = (
    merged
    .dropna(subset=["pfirrman_grade"])
    .groupby("pfirrman_grade")
    .agg(
        total_discs=("failure", "size"),
        failed_discs=("failure", "sum")
    )
    .reset_index()
)

failure_summary["failure_rate_percent"] = (
    failure_summary["failed_discs"]
    / failure_summary["total_discs"]
    * 100
)

print(failure_summary)

13. Calculate Dice distribution by Pfirrmann grade

In [ ]:
dice_summary = (
    merged
    .dropna(subset=["pfirrman_grade"])
    .groupby("pfirrman_grade")["dice"]
    .agg(
        N="count",
        Mean="mean",
        SD="std",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

dice_summary["IQR"] = (
    dice_summary["Q3"]
    - dice_summary["Q1"]
)

print(dice_summary.round(4))

16. Failure rate by Pfirrmann grade

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

grades_present = failure_summary["pfirrman_grade"].astype(int)

ax.bar(
    range(len(failure_summary)),
    failure_summary["failure_rate_percent"]
)

ax.set_xticks(range(len(failure_summary)))

ax.set_xticklabels(
    grades_present.astype(str)
)

ax.set_xlabel("Pfirrmann grade")
ax.set_ylabel("Segmentation failure rate (%)")

ax.set_title(
    "IVD segmentation failure rate by Pfirrmann grade"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Add percentage labels
for i, value in enumerate(
    failure_summary["failure_rate_percent"]
):
    ax.text(
        i,
        value + 0.8,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()

plt.savefig(
    "/path/to/your/data/failure_rate_by_pfirrmann.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "/path/to/your/data/failure_rate_by_pfirrmann.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

17. Statistical analysis
*   Mixed-effects model for Dice

GEE failure model

cluster-aware mixed-effects model (primary omnibus model)

Omnibus test

Estimated marginal means for each grade

Pairwise comparisons with multiplicity adjustment

# Individual structure/level results across all 210 cases HD95 and ASD

In [ ]:
# ============================================================
# SPIDER nnU-Net
# HD95 + ASD analysis
#
# Outputs:
# 1. Overall vertebrae/IVD HD95 + ASD
# 2. Level-wise HD95 + ASD
# 3. Mean ± SD
# 4. Median
# 5. Q1 / Q3
# 6. P90
# 7. N
# 8. Five-fold summaries
# 9. Supplementary boxplots
# ============================================================

Define path

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk
from medpy.metric import binary
import matplotlib.pyplot as plt

# ============================================================
# FILE / DIRECTORY PATHS
# ============================================================

FILE_PATH = "/path/to/your/data/Original_dice_all_fold.xlsx"

GT_DIR = "/path/to/your/data/Spiderdataset/masks"

PRED_DIR = "/path/to/your/data/Prediction_for_Dice_analysis"

OUTPUT_DIR = "/path/to/your/data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("GT directory:", GT_DIR)
print("Prediction directory:", PRED_DIR)
print("Output directory:", OUTPUT_DIR)

Build the GT and prediction indexes

In [ ]:
# ============================================================
# CREATE GT AND PREDICTION FILE INDEX
# ============================================================

gt_files = glob.glob(os.path.join(GT_DIR, "*_t2.mha"))
pred_files = glob.glob(os.path.join(PRED_DIR, "*.nii"))

print("GT files found:", len(gt_files))
print("Prediction files found:", len(pred_files))

In [ ]:
def extract_case_id(filepath):
    filename = os.path.basename(filepath)

    match = re.search(r'(\d+)', filename)

    if match:
        return int(match.group(1))

    return None

4. Label mapping

In [ ]:
# ============================================================
# ANATOMICAL LABEL MAPPING
# ============================================================

VERTEBRA_MAP = {
    "L1": 5,
    "L2": 4,
    "L3": 3,
    "L4": 2,
    "L5": 1
}

GT_IVD_MAP = {
    "T12/L1": 206,
    "L1/L2": 205,
    "L2/L3": 204,
    "L3/L4": 203,
    "L4/L5": 202,
    "L5/S1": 201
}

PRED_IVD_MAP = {
    "T12/L1": 11,
    "L1/L2": 10,
    "L2/L3": 9,
    "L3/L4": 8,
    "L4/L5": 7,
    "L5/S1": 6
}

5. Metric function

In [ ]:
# ============================================================
# HD95 + ASD METRIC FUNCTION
# ============================================================

def calculate_distance_metrics(gt_mask, pred_mask, spacing):

    gt_mask = np.asarray(gt_mask).astype(bool)
    pred_mask = np.asarray(pred_mask).astype(bool)

    if gt_mask.any() and pred_mask.any():

        dice = binary.dc(pred_mask, gt_mask)

        hd95 = binary.hd95(
            pred_mask,
            gt_mask,
            voxelspacing=spacing,
            connectivity=1
        )

        asd = binary.asd(
            pred_mask,
            gt_mask,
            voxelspacing=spacing,
            connectivity=1
        )

        return dice, hd95, asd

    return np.nan, np.nan, np.nan

6. Calculate the level-wise results

In [ ]:
# ============================================================
# LEVEL-WISE HD95 + ASD
# ============================================================

level_results = []

total_cases = len(common_cases)

for case_idx, case_id in enumerate(common_cases, start=1):

    print(f"Processing case {case_idx}/{total_cases}: {case_id}")

    # --------------------------------------------------------
    # READ IMAGES ONCE
    # --------------------------------------------------------

    gt_img = sitk.ReadImage(gt_index[case_id])
    pred_img = sitk.ReadImage(pred_index[case_id])

    gt = sitk.GetArrayFromImage(gt_img)
    pred = sitk.GetArrayFromImage(pred_img)

    # --------------------------------------------------------
    # CHECK SHAPE
    # --------------------------------------------------------

    if gt.shape != pred.shape:
        print(
            f"  WARNING: shape mismatch "
            f"{gt.shape} vs {pred.shape}"
        )
        continue

    # NumPy array order = (z, y, x)
    spacing = tuple(reversed(gt_img.GetSpacing()))

    # --------------------------------------------------------
    # VERTEBRAE
    # --------------------------------------------------------

    for level, label in VERTEBRA_MAP.items():

        gt_mask = (gt == label)
        pred_mask = (pred == label)

        dice, hd95, assd = calculate_distance_metrics(
            gt_mask,
            pred_mask,
            spacing
        )

        level_results.append({
            "case_id": case_id,
            "structure_type": "Vertebra",
            "level": level,
            "Dice": dice,
            "HD95_mm": hd95,
            "ASD_mm": assd
        })

    # --------------------------------------------------------
    # IVDS
    # --------------------------------------------------------

    for level in GT_IVD_MAP:

        gt_mask = (gt == GT_IVD_MAP[level])
        pred_mask = (pred == PRED_IVD_MAP[level])

        dice, hd95, assd = calculate_distance_metrics(
            gt_mask,
            pred_mask,
            spacing
        )

        level_results.append({
            "case_id": case_id,
            "structure_type": "IVD",
            "level": level,
            "Dice": dice,
            "HD95_mm": hd95,
            "ASD_mm": assd
        })

    # --------------------------------------------------------
    # SAVE EVERY 10 CASES
    # --------------------------------------------------------

    if case_idx % 10 == 0:

        temp_df = pd.DataFrame(level_results)

        temp_file = os.path.join(
            OUTPUT_DIR,
            "levelwise_HD95_ASD_progress.csv"
        )

        temp_df.to_csv(temp_file, index=False)

        print(
            f"  Saved progress: {len(temp_df)} rows"
        )


# ============================================================
# FINAL DATAFRAME
# ============================================================

level_df = pd.DataFrame(level_results)

print("\n========================================")
print("COMPLETED")
print("========================================")
print("Total rows:", len(level_df))

display(level_df.head())

7. Calculate statistics

In [ ]:
# ============================================================
# MEDIAN / IQR / P90
# ============================================================

def summarize_metric(series):

    series = series.dropna()

    return pd.Series({
        "N": len(series),
        "Mean": series.mean(),
        "SD": series.std(),
        "Median": series.median(),
        "Q1": series.quantile(0.25),
        "Q3": series.quantile(0.75),
        "P90": series.quantile(0.90)
    })


hd95_summary = (
    level_df
    .groupby(["structure_type", "level"])["HD95_mm"]
    .apply(summarize_metric)
    .unstack()
    .reset_index()
)

asd_summary = (
    level_df
    .groupby(["structure_type", "level"])["ASSD_mm"]
    .apply(summarize_metric)
    .unstack()
    .reset_index()
)

print("HD95:")
display(hd95_summary)

print("ASSD:")
display(assd_summary)

In [ ]:
# STEP 3: Overall and level-wise HD95/ASD distributions

# Keep only observations with valid distance metrics
valid_distance_df = df.dropna(subset=["HD95_mm", "ASD_mm"]).copy()

print("Total valid distance observations:", len(valid_distance_df))
print("Total excluded observations:", len(df) - len(valid_distance_df))


# ---------------------------------------------------------
# 3A. Overall distribution by structure type
# ---------------------------------------------------------

overall_distribution = (
    valid_distance_df
    .groupby("structure_type")
    .agg(
        N=("HD95_mm", "count"),

        HD95_mean=("HD95_mm", "mean"),
        HD95_SD=("HD95_mm", "std"),
        HD95_median=("HD95_mm", "median"),
        HD95_Q1=("HD95_mm", lambda x: x.quantile(0.25)),
        HD95_Q3=("HD95_mm", lambda x: x.quantile(0.75)),
        HD95_P90=("HD95_mm", lambda x: x.quantile(0.90)),

        ASD_mean=("ASD_mm", "mean"),
        ASD_SD=("ASD_mm", "std"),
        ASD_median=("ASD_mm", "median"),
        ASD_Q1=("ASD_mm", lambda x: x.quantile(0.25)),
        ASD_Q3=("ASD_mm", lambda x: x.quantile(0.75)),
        ASD_P90=("ASD_mm", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

overall_distribution["HD95_IQR"] = (
    overall_distribution["HD95_Q3"]
    - overall_distribution["HD95_Q1"]
)

overall_distribution["ASD_IQR"] = (
    overall_distribution["ASD_Q3"]
    - overall_distribution["ASD_Q1"]
)

print("\nOVERALL DISTRIBUTION")
display(overall_distribution)


# ---------------------------------------------------------
# 3B. Level-wise distribution
# ---------------------------------------------------------

level_distribution = (
    valid_distance_df
    .groupby(["structure_type", "level"])
    .agg(
        N=("HD95_mm", "count"),

        HD95_mean=("HD95_mm", "mean"),
        HD95_SD=("HD95_mm", "std"),
        HD95_median=("HD95_mm", "median"),
        HD95_Q1=("HD95_mm", lambda x: x.quantile(0.25)),
        HD95_Q3=("HD95_mm", lambda x: x.quantile(0.75)),
        HD95_P90=("HD95_mm", lambda x: x.quantile(0.90)),

        ASD_mean=("ASD_mm", "mean"),
        ASD_SD=("ASD_mm", "std"),
        ASD_median=("ASD_mm", "median"),
        ASD_Q1=("ASD_mm", lambda x: x.quantile(0.25)),
        ASD_Q3=("ASD_mm", lambda x: x.quantile(0.75)),
        ASD_P90=("ASD_mm", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

level_distribution["HD95_IQR"] = (
    level_distribution["HD95_Q3"]
    - level_distribution["HD95_Q1"]
)

level_distribution["ASD_IQR"] = (
    level_distribution["ASD_Q3"]
    - level_distribution["ASD_Q1"]
)

print("\nLEVEL-WISE DISTRIBUTION")
display(level_distribution)